# ✦ Business-SK · render GPU (Google Colab)

Use this when your **laptop is off**: this Colab becomes the render GPU for your posts (product cut-outs + AI backdrops — the same models the laptop runs).

1. In the Studio press **Render with Colab** and copy the session code.
2. Here: **Runtime → Run all** (first time: *Runtime → Change runtime type → T4 GPU*).
3. Paste the code when asked. Keep this tab open while posts render — it keeps running until the session ends (set **STOP_AFTER_IDLE_MINUTES** in step 4 if you want it to stop by itself).

The first start downloads the models (a few minutes). Google decides how much free GPU time you get.

In [ ]:
# 1 · Check for a GPU
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU here — Runtime → Change runtime type → T4 GPU, then Runtime → Run all again.")
print("GPU:", gpu.stdout.strip())

In [ ]:
# 2 · Install the same libraries the laptop uses (about a minute)
%pip install -q "diffusers==0.40.0" "transformers==5.17.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" timm kornia einops

In [ ]:
# 3 · Connect to your Studio with the one-time session code (the token is never shown)
import json, os, urllib.error, urllib.request
STUDIO = "https://140-238-247-18.nip.io"
SESSION_CODE = ""  #@param {type:"string"}
code = SESSION_CODE.strip() or input("Session code from the Studio (e.g. ABCD-2345): ").strip()
req = urllib.request.Request(STUDIO + "/api/gpu/colab/claim", data=json.dumps({"code": code}).encode(),
                             headers={"Content-Type": "application/json"})
try:
    os.environ["GPU_WORKER_TOKEN"] = json.load(urllib.request.urlopen(req, timeout=30))["token"]
except urllib.error.HTTPError as e:
    raise SystemExit("Code not accepted: " + e.read().decode("utf-8", "ignore")[:200])
print("✔ Connected — this Colab is your render GPU for the next 12 hours")

In [ ]:
# 4 · Render (keep this tab open — Colab itself ends idle / long free sessions)
STOP_AFTER_IDLE_MINUTES = 0  #@param {type:"integer"}
# 0 = never stop by itself; e.g. 30 = stop after 30 minutes without work (frees your free GPU time)
urllib.request.urlretrieve("https://raw.githubusercontent.com/Skarthik06/business-sk/main/scripts/gpu_worker.py", "gpu_worker.py")
os.environ.update(SK_API=STUDIO, SK_WORKER_KIND="colab", SK_IDLE_EXIT_SECS=str(max(0, int(STOP_AFTER_IDLE_MINUTES)) * 60))
!python gpu_worker.py